# Connect Chatbox to MFPortal via MCP

This is a screenshot-driven walkthrough for wiring up [Chatbox](https://chatboxai.app) so it can answer natural-language questions about your slice's live metrics, using the **MCP endpoint** your slice gets once it's registered with the MFPortal.

It has two parts:

- **OPTIONAL** -- setting up FABRIC's own hosted LLM ("FABRIC AI") as a model provider in Chatbox, for anyone who doesn't already have a chat model configured.
- The actual **MCP connection** -- adding MFPortal's `mflib` MCP server to Chatbox and testing it.

**Prerequisite**: you've already registered a slice with the portal (see `MFPortal-Quick-Start.ipynb`) and its status card shows an **MCP Endpoint** section with a **Copy config** button. If it doesn't yet, your measurement node probably isn't fully up -- see that notebook's timing note.

------------------
# OPTIONAL: Set up FABRIC AI as a Model Provider
------------------

**Skip this entire section if you already have a working chat model configured in Chatbox** (Claude, OpenAI, Gemini, etc.) -- the MCP connection further down works with any model that supports tool calling, regardless of which provider it comes from.

This section only covers using FABRIC's own hosted LLM service as that model, for anyone who doesn't have one set up yet.

### 1. Create an LLM token

In the [FABRIC Credential Manager](https://cm.fabric-testbed.net/llm), open the **LLM Tokens** tab (you must be a member of the **FABRIC-LLM** project). Give the token a name, pick a duration, then click **Create Token** and copy the generated key -- you'll paste it into Chatbox in step 3 below.

📸 ![FABRIC Credential Manager -- Create LLM Token](images/fabric-credential-manager-llm-token.png)

### 2. Add a Custom Provider in Chatbox

In Chatbox, go to **Settings → Model Provider → Add**, then scroll to **Custom Additions** and choose **Add Custom Provider**.

📸 ![Chatbox -- Add Custom Provider list](images/chatbox-add-custom-provider-list.png)

Name it something recognizable (e.g. `FABRIC AI`), leave **API Mode** as `OpenAI API Compatible`, and click **Add**.

📸 ![Chatbox -- Add provider dialog](images/chatbox-add-provider-dialog.png)

### 3. Configure the provider and fetch its models

Paste the LLM token from step 1 into **API Key**, and set **API Host** to `https://ai.fabric-testbed.net/v1`. Click **Check** to confirm the key works.

📸 ![Chatbox -- FABRIC AI provider configured, no models fetched yet](images/chatbox-fabric-ai-provider-no-models.png)

Click **Fetch** to pull the available models -- they'll appear in the **Model** list.

📸 ![Chatbox -- FABRIC AI provider with models fetched](images/chatbox-fabric-ai-provider-models-fetched.png)

### 4. Enable "Tool use" on the model(s) you'll use with MCP

Click the gear icon next to a model to open **Edit Model**. **"Tool use" must be checked** for that model to be able to call MCP tools -- without it, the MCP server you add further down will be connected but silently ignored by the model. Click **Test Model** to confirm connectivity, then **Save**.

📸 ![Chatbox -- Edit Model with Vision, Reasoning and Tool use enabled](images/chatbox-edit-model-tool-use-enabled.png)

Not every model needs every capability -- for example `nemotron-nano-30b` has no Vision support, but Tool use still works fine:

📸 ![Chatbox -- Edit Model without Vision, but with Tool use enabled](images/chatbox-edit-model-no-vision-tool-use.png)

That's the whole optional section -- you now have a working chat model in Chatbox. The rest of this notebook applies whether you just set this up or were already using a different model.

------------------
# STEP 1: Copy the MCP config from your slice's status page
------------------

Open your slice's status page (the `view_url` returned by `register_slice_with_portal()`, or via "My Slices" after logging in). Once **Meas Node** shows `healthy`, an **MCP Endpoint** section appears with a **Copy config** button. Click it -- this copies a ready-to-paste JSON snippet (the endpoint URL plus a bearer token) to your clipboard.

📸 ![MFPortal slice status page -- MCP Endpoint section with Copy config button](images/mfportal-slice-status-mcp-endpoint.png)

------------------
# STEP 2: Add the MFPortal MCP server in Chatbox
------------------

Open Chatbox and click **Settings** in the left sidebar.

📸 ![Chatbox main window, Settings in the sidebar](images/chatbox-main-window-settings.png)

Select the **MCP** tab. If this is your first custom server, you'll just see an **Add Server** tile under Custom MCP Servers.

📸 ![Chatbox -- MCP settings, empty state](images/chatbox-mcp-settings-empty.png)

Click **Add Server**, then choose **Import from JSON in clipboard** -- since you already copied the config in Step 1, this fills in everything automatically. (If you'd rather enter it by hand, choose **Add Custom Server** instead.)

📸 ![Chatbox -- Add Server menu](images/chatbox-mcp-add-server-menu.png)

Either way you land on the **Edit MCP Server** screen with Name `mflib`, Type `Remote (http/sse)`, the endpoint URL, and an `Authorization: Bearer ...` HTTP header already filled in. Click **Connect** to verify -- a successful connection lists the available tools (`mflib_cpu`, `mflib_memory`, `mflib_network`, `mflib_nodes`, and more). Click **Save**.

📸 ![Chatbox -- Edit MCP Server, connected with tools listed](images/chatbox-edit-mcp-server-connected.png)

Back on the MCP settings page, your server now appears under Custom MCP Servers with a toggle -- make sure it's switched on.

📸 ![Chatbox -- MCP settings with the mflib server added and enabled](images/chatbox-mcp-settings-server-added.png)

------------------
# STEP 3: Enable MCP for a chat and ask a question
------------------

Start a new chat and switch to **Work Mode** (Chat Mode doesn't expose tools). Open the tools panel -- **MCP** should show your connected server, with a badge for how many tools are enabled. Make sure the model you're using has **Tool use** enabled (see the optional section above if you're using FABRIC AI).

📸 ![Chatbox -- Work Mode panel showing the MCP entry](images/chatbox-work-mode-mcp-panel.png)

Now ask something grounded in your slice's real metrics, for example:

- "List the nodes in my slice."
- "Which node is using the most memory?"

The assistant answers using fixed, safe PromQL queries against your own Prometheus data -- not guesses.

## Troubleshooting / FAQ

**The model doesn't seem to use the tools / ignores the MCP server.**
Check that **Tool use** is enabled for that model in its Edit Model dialog (Model Provider settings) -- an MCP server can be connected and enabled and still be ignored if the model itself isn't flagged as tool-capable.

**"No models available" after configuring a provider.**
Click **Fetch** -- models aren't listed until you do. If that still shows nothing, double check the API Key and API Host, then **Check** again.

**The bearer token stopped working.**
The MCP token is tied to a specific slice registration. If you re-registered the slice, go back to the status page and click **Copy config** again -- the token rotates.

**No MCP Endpoint section on my slice's status page.**
The measurement node isn't fully up yet -- see `MFPortal-Quick-Start.ipynb`'s note on first-time setup taking 15–20 minutes.

## Further reading

- `MFPortal-Quick-Start.ipynb` (this directory) -- registering a slice with the portal in the first place.
- `MFPORTAL_METHODS.md` / `PORTAL_REGISTRATION_OPTIONS.md` (mflib repo root) -- full `MFPortal` API reference.